# PIRVN — QLoRA Fine-Tuning on Colab

Fine-tune Qwen2.5-3B to predict Vietnamese product prices.

**Runtime**: GPU (T4) required. Go to Runtime > Change runtime type > T4 GPU.

In [ ]:
!pip install unsloth datasets trl

In [ ]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    use_gradient_checkpointing="unsloth",
)

## Load Training Data

Upload your `train.jsonl` and `val.jsonl` files from `phase4_finetune/` to Colab.

In [ ]:
from datasets import load_dataset
dataset = load_dataset("json", data_files={"train": "train.jsonl", "validation": "val.jsonl"})

print(f"Train examples: {len(dataset['train'])}")
print(f"Validation examples: {len(dataset['validation'])}")
print(f"\nSample keys: {dataset['train'].column_names}")
print(f"\nFirst example:\n{dataset['train'][0]}")

In [ ]:
def format_chat_template(example):
    """
    Format each example's messages list into the model's chat template.
    Expects each example to have a 'messages' field containing a list of
    {role, content} dicts (system/user/assistant turns).
    """
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = dataset.map(format_chat_template, batched=False)

print("Formatted sample:")
print(dataset["train"][0]["text"][:500])

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    args=SFTConfig(
        output_dir="pirvn-pricer-output",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_steps=100,
        logging_steps=50,
        eval_strategy="steps",
        eval_steps=500,
        save_strategy="steps",
        save_steps=500,
        fp16=True,
        max_seq_length=2048,
        seed=42,
    ),
)

In [ ]:
trainer.train()

## Save & Export

In [ ]:
model.save_pretrained_merged("pirvn-pricer-merged", tokenizer)

In [ ]:
model.save_pretrained_gguf("pirvn-pricer-gguf", tokenizer, quantization_method="q4_k_m")

## Quick Test

In [ ]:
# Switch model to inference mode
FastLanguageModel.for_inference(model)

# Sample Vietnamese product description
sample_description = (
    "Ao so mi nam tay dai chat lieu cotton cao cap, "
    "thiet ke co ban don gian, phu hop di lam va di choi."
)

messages = [
    {
        "role": "system",
        "content": (
            "You are a Vietnamese product price estimator. "
            "Given a product description, respond with only the predicted price in VND as a plain integer."
        ),
    },
    {
        "role": "user",
        "content": f"Predict the price for this product: {sample_description}",
    },
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        temperature=0.1,
        do_sample=True,
    )

response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(f"Product: {sample_description}")
print(f"Predicted price: {response.strip()} VND")

In [ ]:
# Optional: push merged model to HuggingFace Hub
# Replace 'your-username' with your HuggingFace username before running.
# You must be logged in: run `huggingface-cli login` or set the HF_TOKEN env var.
model.push_to_hub_merged("your-username/pirvn-pricer", tokenizer)